# Modeling Experiments

This notebook extends the baseline models from `modeling_baselines.ipynb` with more complex model architectures for predicting WNBA player salaries from on-court production metrics.

### Objectives:
1. Load pre-processed train/test artifacts from `data/processed/` (25 selected features).
2. Enforce structural chronological separation using the `WNBALeakageProofSplitter`.
3. Evaluate complex models beyond the baseline tree-based approaches.
4. Assess out-of-fold historical cross-validation performance using the same primary, secondary, and business-focused KPIs as `modeling_baselines.ipynb`.
5. Save fitted models to `models/` for downstream tuning.

### Models Evaluated:
* **Regularized Linear:** `ElasticNet` (combined L1 + L2 regularization)
* **Support Vector:** `SVR` (kernel-based regression, effective on small datasets)
* **Boosting Ensemble:** `GradientBoostingRegressor` (sequential tree boosting, sklearn)
* **Boosting Ensemble:** `XGBRegressor` (extreme gradient boosting)
* **Stacking Ensemble:** `StackingRegressor` (meta-learner combining Ridge + RF + GBM)

## 1. Setup & Imports

In [8]:
import re
import joblib
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, StackingRegressor
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR

from xgboost import XGBRegressor

np.random.seed(26)

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent

processed_dir = repo / "data" / "processed"
models_dir    = repo / "models"
models_dir.mkdir(parents=True, exist_ok=True)

print("Imports OK")

Imports OK


## 2. Load Processed Data

Loads pre-processed artifacts from `data/processed/` produced by `train_test_processed.ipynb`. 

In [9]:
X_train = pd.read_csv(processed_dir / "X_train_processed.csv")
y_train = pd.read_csv(processed_dir / "y_train.csv")["salary"]
X_test  = pd.read_csv(processed_dir / "X_test_2025_processed.csv")
y_test  = pd.read_csv(processed_dir / "y_test_2025.csv")["salary"]
lookup_train = pd.read_csv(processed_dir / "player_lookup_train.csv")
lookup_test  = pd.read_csv(processed_dir / "player_lookup_test_2025.csv")

# Derive feature names directly from X_train (avoids stale feature_names.csv)
feature_names = X_train.columns.tolist()
ohe_cols = [c for c in feature_names if c.startswith("group_")]
num_cols = [c for c in feature_names if c not in ohe_cols]

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Numeric features: {len(num_cols)}, OHE features: {len(ohe_cols)}")
print(f"Features: {feature_names}")

X_train: (742, 25), X_test: (223, 25)
Numeric features: 20, OHE features: 5
Features: ['avail_rate', 'blk', 'fg', 'fg_per_g', 'fga', 'ft', 'fta', 'g', 'mp', 'pca1', 'pts', 'pts_rookie', 'pts_vet', 'pts_hardship', 'start_rate', 'team_min', 'tov', 'ws_rookie', 'ws_vet', 'ws_hardship', 'group_controlled', 'group_hardship', 'group_rookie', 'group_unknown', 'group_veteran']


## 3. Chronological CV Splitter

In [10]:
# Attach year from lookup so we can generate chronological folds
X_train_with_year = X_train.copy()
X_train_with_year["year"] = lookup_train["year"].values

def generate_cv_folds(X_with_year):
    """Chronological CV folds — same logic as WNBALeakageProofSplitter."""
    sorted_years = sorted(X_with_year["year"].unique())
    if len(sorted_years) < 2:
        raise ValueError("Insufficient years for CV folds.")
    X_reset = X_with_year.reset_index(drop=True)
    for i in range(1, len(sorted_years)):
        train_years = sorted_years[:i]
        val_year    = sorted_years[i]
        train_idx = X_reset[X_reset["year"].isin(train_years)].index.tolist()
        val_idx   = X_reset[X_reset["year"] == val_year].index.tolist()
        yield np.array(train_idx), np.array(val_idx)

# Verify folds
for fold, (tr_idx, val_idx) in enumerate(generate_cv_folds(X_train_with_year), start=1):
    tr_yr  = X_train_with_year.iloc[tr_idx]["year"].max()
    val_yr = X_train_with_year.iloc[val_idx]["year"].min()
    assert tr_yr < val_yr, f"Temporal leakage in fold {fold}"
    print(f"Fold {fold}: train up to {tr_yr}, val = {val_yr}")

Fold 1: train up to 2021, val = 2022
Fold 2: train up to 2022, val = 2023
Fold 3: train up to 2023, val = 2024


## 4. Complex Model Definitions


In [11]:
complex_models = {
    "ElasticNet": ElasticNet(
        alpha=1.0, l1_ratio=0.5, max_iter=5000, random_state=26
    ),
    "SVR (RBF Kernel)": SVR(
        kernel="rbf", C=1.0, epsilon=0.1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200, max_depth=3, learning_rate=0.1, random_state=26
    ),
    "Stacking Ensemble": StackingRegressor(
        estimators=[
            ("ridge", Ridge(alpha=1.0)),
            ("rf",    RandomForestRegressor(n_estimators=150, min_samples_leaf=5, random_state=26)),
            ("gbm",   GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=26)),
        ],
        final_estimator=Ridge(alpha=1.0),
        cv=3,
    ),
}

complex_models["XGBoost"] = XGBRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,        random_state=26, verbosity=0
)

print(f"Models registered: {list(complex_models.keys())}")

Models registered: ['ElasticNet', 'SVR (RBF Kernel)', 'Gradient Boosting', 'Stacking Ensemble', 'XGBoost']


## 5. Chronological Cross-Validation

Same evaluation loop as `modeling_baselines.ipynb`. Two KPIs reported:
- **Primary:** RMSE
- **Secondary:** MAPE

 **Note:** CPWS MAE is skipped here. Raw `ws` (win shares) was dropped during feature selection and is not available in `X_train`. Only interaction terms `ws_rookie`, `ws_vet`, `ws_hardship` were retained. To re-enable CPWS, we need to add `ws` to `player_lookup_train.csv`.


In [12]:
model_cv_summary = []

for name, model in complex_models.items():
    fold_rmses = []
    fold_mapes = []

    for fold, (train_idx, val_idx) in enumerate(generate_cv_folds(X_train_with_year), start=1):
        X_tr,  y_tr  = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_val, y_val = X_train.iloc[val_idx],   y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)

        # Primary KPI: RMSE
        fold_rmses.append(np.sqrt(mean_squared_error(y_val, y_pred)))

        # Secondary KPI: MAPE
        fold_mapes.append(mean_absolute_percentage_error(y_val, y_pred))

    # Note: CPWS MAE skipped — raw 'ws' was dropped during feature selection and
    # is not available in X_train. Add 'ws' to player_lookup_train.csv to re-enable.
    model_cv_summary.append({
        "Model Architecture":  name,
        "Primary KPI: RMSE":   np.mean(fold_rmses),
        "Secondary KPI: MAPE": f"{np.mean(fold_mapes) * 100:.2f}%",
    })
    print(f"{name}: RMSE=${np.mean(fold_rmses):,.0f}")

results_df = pd.DataFrame(model_cv_summary)


ElasticNet: RMSE=$40,760
SVR (RBF Kernel): RMSE=$71,805
Gradient Boosting: RMSE=$43,876
Stacking Ensemble: RMSE=$39,441
XGBoost: RMSE=$43,317


In [13]:
print("CV Experiment Performance")
print(results_df.to_string(index=False))

CV Experiment Performance
Model Architecture  Primary KPI: RMSE Secondary KPI: MAPE
        ElasticNet       40759.631364             219.61%
  SVR (RBF Kernel)       71805.016801             378.13%
 Gradient Boosting       43875.823191             210.46%
 Stacking Ensemble       39441.479332             167.73%
           XGBoost       43316.572133             213.71%
